In [1]:
import os, sys
# Cache paths
os.environ['HF_HOME'] = '/mnt/beegfs/msaxena4/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/mnt/beegfs/msaxena4/hf_cache'

# Test write permission
test_file_path = os.path.join(os.environ['TRANSFORMERS_CACHE'], 'text.txt')
try:
    with open(test_file_path, 'w') as f:
        f.write('This is a test.')
    print('Write successful!')
except Exception as e:
    print('Error writing to directory: ', e)

Write successful!


In [2]:
import pandas as pd
import numpy as np
import random
import math
import ast
import torch
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer, BertForNextSentencePrediction

/mnt/beegfs/msaxena4/miniconda/lib/python3.12/site-packages/transformers/utils/hub.py:111: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
NUM_NARRATIVES = 100

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForNextSentencePrediction.from_pretrained('bert-base-uncased')

In [4]:
def get_NSP_prob(model, tokenizer, s1, s2):
    encoding = tokenizer(s1, s2, return_tensors='pt')
    outputs = model(**encoding)
    logits = outputs.logits
    probs = torch.softmax(logits, dim=1)
    return probs[0, 0].item()   

In [5]:
def get_each_candidate_NSP(row, model, tokenizer):
    sentences = row["modified_sentences"]
    myth_idx = int(row["myth_sentence_idx"])

    prev_idx = row["prev_sentence_idx"]
    next_idx = row["next_sentence_idx"]

    prev_prob = float('nan')
    next_prob = float('nan')

    # Previous pair
    if not math.isnan(prev_idx):
        prev_idx = int(prev_idx)
        s1 = sentences[prev_idx]
        s2 = sentences[myth_idx]
        prev_prob = get_NSP_prob(model, tokenizer, s1, s2)

    # Next pair
    if not math.isnan(next_idx):
        next_idx = int(next_idx)
        s1 = sentences[myth_idx]
        s2 = sentences[next_idx]
        next_prob = get_NSP_prob(model, tokenizer, s1, s2)

    probs = [p for p in [prev_prob, next_prob] if not math.isnan(p)]
    mean_prob = sum(probs) / len(probs) if probs else float('nan')

    return prev_prob, next_prob, mean_prob


In [6]:
def get_all_candidate_NSP(df):
    results = []
    for idx, row in df.iterrows():
        prev_prob, next_prob, mean_prob = get_each_candidate_NSP(
            row, model, tokenizer
        )
        results.append({
            "narrative_idx": row["narrative_idx"],
            "StoryOutline": row["StoryOutline"],
            "original_narrative": row["original_narrative"],
            "myth_type": row["myth_type"],
            "myth_variation": row["myth_variation"],
            "dose": row["dose"],
            "myth_detail": row["myth_detail"],
            "modified_narrative": row["modified_narrative"],
            "modified_sentences": row["modified_sentences"],
            "insertion_idx": row["myth_sentence_idx"],
            "prev_prob": prev_prob,
            "next_prob": next_prob,
            "mean_prob": mean_prob
        })
    
    return pd.DataFrame(results)

In [7]:
gemini_df = pd.read_csv('Results/Gemini_candidates.csv', converters={"modified_sentences": ast.literal_eval})
llama_df = pd.read_csv('Results/Llama_candidates.csv', converters={"modified_sentences": ast.literal_eval})
mistral_df = pd.read_csv('Results/Mistral_candidates.csv', converters={"modified_sentences": ast.literal_eval})

In [8]:
# (gemini_df['index'] == gemini_df['myth_sentence_idx']).all()

In [9]:
gemini_out = get_all_candidate_NSP(gemini_df)
# llama_out = get_all_candidate_NSP(llama_df)
# mistral_out = get_all_candidate_NSP(mistral_df)

KeyboardInterrupt: 

In [ ]:
# gemini_out.to_csv("Results/Gemini_candidates_NSP.csv", index=False)
# llama_out.to_csv("Results/Llama_candidates_NSP.csv", index=False)
# mistral_out.to_csv("Results/Mistral_candidates_NSP.csv", index=False)

In [ ]:
# np.nan is np.nan